# NDVI for Arica y Parinacota, Chile (Sentinel-2, 2024)

1. **Objetivo del notebook**
- Descargar composiciones NDVI por semana epidemiológica.

2. **Decisiones metodológicas tomadas en la construcción del notebook**
- Usar SE1-SE52, máscara de nubes y ruta externa canónica.

3. **Archivos generados en el notebook y ubicación de guardado**
- GeoTIFF por estadístico y semana en TESIS_NDVI_ROOT o D:/GeoTIFF NDVI.

Cada sección Markdown posterior explica brevemente qué hace el código que la sigue.


This notebook downloads Sentinel-2 surface reflectance imagery from Google Earth Engine,
masks clouds and shadows using the SCL band, computes NDVI = (B8 - B4) / (B8 + B4),
builds epidemiological-week statistic composites, and saves one single-band GeoTIFF per statistic and week to statistic-specific folders under `data/ndvi/ndvi_semanal/`.

**Area of interest:** Region of Arica y Parinacota.
Bounding box (lon, lat): -70.40, -19.25, -68.90, -17.55.

**Time range:** 2023-12-31 through 2024-12-28 (52 epidemiological-week composites for Chile 2024).

**Output:** `data/ndvi/ndvi_semanal/media_semanal/ndvi_arica_y_parinacota_2024_SE##_mean.tif`, `..._min.tif`, `..._max.tif`, `..._std.tif`; single-band float32, EPSG:32719, 10 m pixels.


In [ ]:
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
while not (PROJECT_ROOT / "src").exists() and PROJECT_ROOT != PROJECT_ROOT.parent:
    PROJECT_ROOT = PROJECT_ROOT.parent

sys.path.append(str(PROJECT_ROOT))

from src.paths import *
ensure_directories()


## 1. Imports and configuration

Carga las librerías, las rutas centrales y la configuración necesaria para ejecutar el flujo.


In [ ]:
import ee
import geemap
import rasterio
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from pathlib import Path

# Earth Engine Cloud project id. Replace with your own before the first run.
GEE_PROJECT = 'ndvi-data-for-chile' # Replace with your own GEE project ID. You can find this in the Earth Engine Code Editor under "Assets" > "Projects". It is required for exporting data from Earth Engine to Google Drive or Cloud Storage.

# Area of interest: [west, south, east, north] in degrees
AOI_BBOX = [-70.38, -19.23, -68.90, -17.49]

# Time range
START = '2023-12-31'
END = '2024-12-29'

# Sentinel-2 native resolution in meters
SCALE = 10 # Sentinel-2 has 10m, 20m, and 60m bands. We will use the 10m bands for NDVI.

# Drop scenes with more than this percent of cloud cover before per-pixel masking
MAX_CLOUD_PCT = 60 

# Output directory (relative to this notebook)
OUT_DIR = require_ndvi_external_dir() / 'ndvi_semanal'
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f'Output directory: {OUT_DIR.resolve()}')


## 2. Authenticate to Earth Engine

The first run opens a browser window. After that the credentials are cached in `~/.config/earthengine/`.


In [ ]:
try:
    ee.Initialize(project=GEE_PROJECT)
    print('Earth Engine initialized.')
except Exception:
    ee.Authenticate()
    ee.Initialize(project=GEE_PROJECT)
    print('Earth Engine authenticated and initialized.')


## 3. Define the area of interest

Define la región de interés mediante su extensión espacial y los parámetros de exportación.


In [ ]:
aoi = ee.Geometry.Rectangle(AOI_BBOX)

center_lat = (AOI_BBOX[1] + AOI_BBOX[3]) / 2
center_lon = (AOI_BBOX[0] + AOI_BBOX[2]) / 2

m = geemap.Map(center=[center_lat, center_lon], zoom=10)
m.addLayer(aoi, {'color': 'red'}, 'AOI')
m


## 4. Build a cloud-masked Sentinel-2 collection

SCL classes that get masked: 3 (cloud shadow), 8 (medium-prob cloud), 9 (high-prob cloud), 10 (thin cirrus), 11 (snow / ice).


In [ ]:
SCL_BAD_CLASSES = [3, 8, 9, 10, 11]

def mask_s2_scl(image):
    scl = image.select('SCL')
    mask = scl.neq(SCL_BAD_CLASSES[0])
    for cls in SCL_BAD_CLASSES[1:]:
        mask = mask.And(scl.neq(cls))
    return image.updateMask(mask)

s2 = (ee.ImageCollection('COPERNICUS/S2_SR_HARMONIZED')
      .filterBounds(aoi)
      .filterDate(START, END)
      .filter(ee.Filter.lt('CLOUDY_PIXEL_PERCENTAGE', MAX_CLOUD_PCT))
      .map(mask_s2_scl))

print(f'Scenes after cloud filter: {s2.size().getInfo()}')


## 5. Compute NDVI

Calcula la banda NDVI a partir de las bandas espectrales seleccionadas.


In [ ]:
def add_ndvi(image):
    ndvi = image.normalizedDifference(['B8', 'B4']).rename('NDVI')
    return ndvi.copyProperties(image, ['system:time_start'])

ndvi_collection = s2.map(add_ndvi)


## 6. Build epidemiological-week NDVI statistic composites

Agrupa las imágenes por semana epidemiológica y calcula los estadísticos NDVI solicitados.


In [ ]:
EPI_YEAR = 2024
EPI_WEEK_1_START = pd.Timestamp('2023-12-31')
N_EPI_WEEKS = 52

epi_weeks = pd.DataFrame({
    'epi_year': EPI_YEAR,
    'epi_week': range(1, N_EPI_WEEKS + 1),
})
epi_weeks['week_start'] = EPI_WEEK_1_START + pd.to_timedelta((epi_weeks['epi_week'] - 1) * 7, unit='D')
epi_weeks['week_end'] = epi_weeks['week_start'] + pd.Timedelta(days=6)

print(f"Building {len(epi_weeks)} epidemiological-week composites "
      f"from {epi_weeks['week_start'].min().date()} to {epi_weeks['week_end'].max().date()}.")

def epi_week_stats(row):
    start = ee.Date(row.week_start.strftime('%Y-%m-%d'))
    end = ee.Date((row.week_end + pd.Timedelta(days=1)).strftime('%Y-%m-%d'))
    week_collection = ndvi_collection.filterDate(start, end)
    img = (week_collection
           .reduce(ee.Reducer.mean()
                   .combine(ee.Reducer.min(), sharedInputs=True)
                   .combine(ee.Reducer.max(), sharedInputs=True)
                   .combine(ee.Reducer.stdDev(), sharedInputs=True))
           .rename(['ndvi_mean', 'ndvi_min', 'ndvi_max', 'ndvi_std'])
           .clip(aoi)
           .set({'system:time_start': start.millis(),
                 'epi_year': int(row.epi_year),
                 'epi_week': int(row.epi_week),
                 'week_start': row.week_start.strftime('%Y-%m-%d'),
                 'week_end': row.week_end.strftime('%Y-%m-%d')}))
    return img

weekly_images = {
    f"{int(row.epi_year)}_SE{int(row.epi_week):02d}": epi_week_stats(row)
    for row in epi_weeks.itertuples(index=False)
}


## 7. Export each composite to a local GeoTIFF

`geemap.download_ee_image` automatically tiles requests larger than the synchronous size limit, so no Drive round-trip is needed. The loop is resumable: existing files are skipped. Each statistic is exported as a separate single-band GeoTIFF.


In [ ]:
# Sufijo del archivo de salida -> banda interna de Earth Engine.
stats_to_export = {
    'mean': 'ndvi_mean',
    'min': 'ndvi_min',
    'max': 'ndvi_max',
    'std': 'ndvi_std',
}

# Sufijo del archivo de salida -> subcarpeta donde se guarda.
stat_output_dirs = {
    'mean': 'media_semanal',
    'min': 'minimo_semanal',
    'max': 'maximo_semanal',
    'std': 'desvest_semanal',
}

for stat_dir in stat_output_dirs.values():
    (OUT_DIR / stat_dir).mkdir(parents=True, exist_ok=True)

for week_label, img in weekly_images.items():
    for stat_label, band_name in stats_to_export.items():
        stat_dir = stat_output_dirs[stat_label]
        out_path = OUT_DIR / stat_dir / f'ndvi_arica_y_parinacota_{week_label}_{stat_label}.tif'
        if out_path.exists():
            print(f'  skip {out_path.relative_to(OUT_DIR)} (already exists)')
            continue
        print(f'  download {out_path.relative_to(OUT_DIR)}')
        try:
            geemap.download_ee_image(
                image=img.select(band_name),
                filename=str(out_path),
                region=aoi,
                scale=SCALE,
                crs='EPSG:32719',
            )
        except Exception as exc:
            print(f'    failed: {exc}')

print('Done.')


## 8. Output check

Comprueba la existencia y el nombre de los GeoTIFF esperados en la ruta externa.


In [ ]:
expected_files = [
    OUT_DIR / stat_output_dirs[stat] / f"ndvi_arica_y_parinacota_2024_SE{week:02d}_{stat}.tif"
    for week in range(1, N_EPI_WEEKS + 1)
    for stat in stats_to_export
]
existing_files = [path for path in expected_files if path.exists()]
missing_files = [path.relative_to(OUT_DIR) for path in expected_files if not path.exists()]

print(f"Archivos esperados: {len(expected_files)}")
print(f"Archivos encontrados: {len(existing_files)}")
print(f"Archivos faltantes: {len(missing_files)}")

if missing_files:
    print("Primeros faltantes:")
    for name in missing_files[:10]:
        print(f"  {name}")


## Outputs

The notebook writes one single-band GeoTIFF per statistic and epidemiological week to statistic-specific folders under `data/ndvi/ndvi_semanal/`, e.g.

```
data/ndvi/ndvi_semanal/media_semanal/ndvi_arica_y_parinacota_2024_SE01_mean.tif
data/ndvi/ndvi_semanal/minimo_semanal/ndvi_arica_y_parinacota_2024_SE01_min.tif
data/ndvi/ndvi_semanal/maximo_semanal/ndvi_arica_y_parinacota_2024_SE01_max.tif
data/ndvi/ndvi_semanal/desvest_semanal/ndvi_arica_y_parinacota_2024_SE01_std.tif
...
data/ndvi/ndvi_semanal/desvest_semanal/ndvi_arica_y_parinacota_2024_SE52_std.tif
```

Each raster is single-band float32, EPSG:32719 (UTM zone 19S), 10 m pixel size. The statistic is indicated by the filename suffix: `mean`, `min`, `max`, or `std`.


In [ ]:
print("Path migration check completed.")
